# Salah Posture Detection - Data Exploration

Explore collected sensor data for prayer posture classification.

**Steps:**
1. Load JSONL data files
2. Check class distribution
3. Visualize sensor patterns per posture
4. Feature engineering analysis
5. Assess data quality and readiness for training

In [ ]:
import sys
sys.path.insert(0, '../salah_model')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from feature_engineering import load_jsonl_files, extract_window_features, POSTURE_LABELS

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Data

In [ ]:
DATA_DIR = '../data'  # Copy JSONL files from phone here
samples = load_jsonl_files(DATA_DIR)
print(f'Total samples: {len(samples)}')

## 2. Class Distribution

In [ ]:
posture_counts = {}
for s in samples:
    p = s.get('posture', 'UNKNOWN')
    posture_counts[p] = posture_counts.get(p, 0) + 1

fig, ax = plt.subplots(figsize=(10, 5))
postures = list(posture_counts.keys())
counts = [posture_counts[p] for p in postures]
bars = ax.bar(postures, counts, color=['#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336', '#607D8B'][:len(postures)])
ax.set_title('Samples per Posture')
ax.set_ylabel('Count')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(count), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nPer-posture counts:')
for p, c in sorted(posture_counts.items()):
    print(f'  {p}: {c} windows ({c * 0.1:.1f}s of data)')

## 3. Sensor Patterns per Posture

Visualize pitch, roll, and accelerometer magnitude distributions for each posture.

In [ ]:
# Collect pitch, roll, accel_magnitude per posture
data_by_posture = {}
for s in samples:
    p = s.get('posture', 'UNKNOWN')
    if p not in data_by_posture:
        data_by_posture[p] = {'pitch': [], 'roll': [], 'accel_mag': [], 'gyro_mag': []}
    data_by_posture[p]['pitch'].append(s.get('pitch', 0))
    data_by_posture[p]['roll'].append(s.get('roll', 0))
    data_by_posture[p]['accel_mag'].append(s.get('accel_magnitude', 0))
    data_by_posture[p]['gyro_mag'].append(s.get('gyro_magnitude', 0))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, metric, title in zip(axes.flat,
    ['pitch', 'roll', 'accel_mag', 'gyro_mag'],
    ['Pitch (degrees)', 'Roll (degrees)', 'Accel Magnitude (m/s²)', 'Gyro Magnitude (rad/s)']):
    for p in POSTURE_LABELS:
        if p in data_by_posture:
            vals = data_by_posture[p][metric]
            ax.hist(vals, bins=50, alpha=0.5, label=p, density=True)
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle('Sensor Distributions by Posture', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Feature Analysis

Extract features and check separability between postures.

In [ ]:
from feature_engineering import extract_single_window_features, POSTURE_TO_INDEX

X, y = extract_single_window_features(samples)
print(f'Feature matrix: {X.shape}')
print(f'Labels: {y.shape}')

# Feature names
feature_names = [
    'accel_mean_x', 'accel_mean_y', 'accel_mean_z',
    'accel_std_x', 'accel_std_y', 'accel_std_z',
    'accel_mag_mean', 'accel_mag_var',
    'gyro_mean_x', 'gyro_mean_y', 'gyro_mean_z',
    'gyro_std_x', 'gyro_std_y', 'gyro_std_z',
    'gyro_mag_mean', 'gyro_mag_var',
    'pitch_mean', 'pitch_var', 'roll_mean', 'roll_var',
    'accel_min', 'accel_max', 'gyro_min', 'gyro_max',
    'pitch', 'roll', 'accel_magnitude', 'gyro_magnitude',
    'accel_energy', 'gyro_energy'
]

df = pd.DataFrame(X, columns=feature_names)
df['posture'] = [POSTURE_LABELS[i] for i in y]
print('\nFeature statistics per posture:')
print(df.groupby('posture')[['pitch', 'roll', 'accel_magnitude', 'gyro_magnitude']].describe().round(2))

## 5. Pitch vs Roll Scatter Plot

The most discriminative features for salah postures are pitch and roll.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
colors = {'QIYAM': '#2196F3', 'RUKU': '#FF9800', 'SUJUD': '#4CAF50', 'JALSA': '#9C27B0', 'TASHAHHUD': '#F44336'}

for posture in POSTURE_LABELS:
    mask = df['posture'] == posture
    ax.scatter(df.loc[mask, 'pitch'], df.loc[mask, 'roll'],
               alpha=0.4, s=20, label=posture, color=colors.get(posture, '#999'))

ax.set_xlabel('Pitch (degrees)')
ax.set_ylabel('Roll (degrees)')
ax.set_title('Pitch vs Roll by Posture')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Data Quality Check

In [ ]:
print('=== Data Quality Report ===')
print(f'Total samples: {len(samples)}')
print(f'Unique sessions: {len(set(s.get("session_id", "") for s in samples))}')
print()

min_target = 500
print(f'Target: {min_target} windows per posture\n')
ready = True
for p in POSTURE_LABELS:
    count = posture_counts.get(p, 0)
    status = 'OK' if count >= min_target else f'NEED {min_target - count} MORE'
    if count < min_target:
        ready = False
    print(f'  {p:12s}: {count:5d} windows  [{status}]')

print(f'\nTraining readiness: {"READY" if ready else "NOT READY - collect more data"}')

if not ready:
    total_needed = sum(max(0, min_target - posture_counts.get(p, 0)) for p in POSTURE_LABELS)
    print(f'Total additional windows needed: ~{total_needed}')
    print(f'Estimated additional recording time: ~{total_needed * 0.1 / 60:.1f} minutes')